In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [2]:
transform = transforms.ToTensor()

In [3]:
train_data = MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)


100%|███████████████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:04<00:00, 2.16MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 92.1kB/s]
100%|████████████████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:02<00:00, 725kB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<?, ?B/s]


In [4]:
test_data = MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

In [5]:
train_loader= DataLoader(
    train_data,
    batch_size=64,
    shuffle=True)



test_loader = DataLoader(test_data,
                         batch_size=64,
                         shuffle=False)

In [6]:
import torch
import torch.nn as nn

class DigitalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
            nn.ReLU(),
            nn.Linear(10, 10),
        )

    def forward(self, x):
        # x = x.view(x.size(0), -1)  # Flatten image
        return self.net(x)

model = DigitalModel()

print(model)

DigitalModel(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=10, bias=True)
  )
)


In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        # Flatten 28x28 image -> 784
        images = images.view(images.size(0), -1)

        # Move to device
        # images = images.to(device)
        # labels = labels.to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = loss_fn(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )


model.eval()
correct=0
total=0

with torch.no_grad():
    for images, labels in test_loader:
        # Flatten 28x28 image -> 784
        images = images.view(images.size(0), -1) # Corrected: 'images' instead of 'image'
        
        # Move to device (if applicable, uncomment if needed)
        # images = images.to(device)
        # labels = labels.to(device)

        outputs = model(images) # Corrected: removed trailing comma
        _, predicted = torch.max(outputs.data, 1) # Corrected: uncommented and fixed
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


torch.save(model.state_dict(), "model.pth")
print('model saved')


index= 0
image , true_label = test_data[index] # Corrected: used [] for dataset access

Epoch [1/5] Loss: 0.4680
Epoch [2/5] Loss: 0.1913
Epoch [3/5] Loss: 0.1345
Epoch [4/5] Loss: 0.1058
